***

* [总目录](../0_Introduction/0_introduction.ipynb)
* [术语表](../0_Introduction/1_glossary.ipynb)
* [7. 观测系统](7_0_introduction.ipynb)
    * 上一节： [7.1 琼斯符号](7_1_jones_notation.ipynb)
    * 下一节： [7.3 模拟电子学](7_3_analogue.ipynb)

***

导入标准模块:

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

plt.rcParams['figure.max_open_warning'] = 0

NOTEBOOK_DIR = Path('7_Observing_Systems') if Path('7_Observing_Systems').exists() else Path('.')
NOTEBOOK_DIR = NOTEBOOK_DIR.resolve()
STYLE_PATH = NOTEBOOK_DIR.parent / 'style' / 'course.css'
TOGGLE_PATH = NOTEBOOK_DIR.parent / 'style' / 'code_toggle.html'

try:
    from IPython.display import HTML
except ImportError:
    def HTML(*args, **kwargs):
        return None

if STYLE_PATH.exists():
    try:
        HTML(STYLE_PATH.read_text(encoding='utf-8'))
    except Exception:
        pass

本节以下示例全部使用 notebook 内生成的亮度矩阵、随机电场样本和离散天空模型，因此不依赖旧版 `rime_lib.py` 或外部图像。重点将放在四个层次上：相关矩阵怎样从电压样本中估计出来，单点源的几何相位怎样进入可见度，离散天空求和如何过渡到全天 RIME，以及 Jones 语言怎样映射到 Mueller 语言。

In [ ]:
if TOGGLE_PATH.exists():
    try:
        HTML(TOGGLE_PATH.read_text(encoding='utf-8'))
    except Exception:
        pass

***

## 7.2 射电干涉测量方程（RIME） <a id='instrum:sec:rime'></a>

上一节已经建立了 Jones 向量、亮度矩阵和 Jones 链的语言。RIME 的工作，就是把这些对象正式放进干涉仪的可见度测量里。最核心的想法其实非常简单：干涉仪并不是直接测量“天空亮度”，而是测量不同天线处电压信号之间的二阶相关；而这些电压信号本身，又是天空电场经过传播与仪器 Jones 链之后的结果。

因此，RIME 不是一条孤立的公式，而是一种系统建模框架：

* 源辐射用亮度矩阵 $\mathbf{B}$ 描述；
* 传播和仪器效应用 Jones 矩阵描述；
* 相关器输出用可见度矩阵 $\mathbf{V}_{pq}$ 描述；
* 多源天空则通过对各个方向上的贡献做求和或积分得到。

本节先从最简化的二元干涉仪出发，再逐步走向单点源、离散天空和 Mueller 表示。

In [ ]:
def brightness_matrix(I, Q=0.0, U=0.0, V=0.0):
    return 0.5 * np.array(
        [[I + Q, U + 1j * V], [U - 1j * V, I - Q]],
        dtype=complex,
    )



def stokes_from_brightness(B):
    xx = B[0, 0]
    yy = B[1, 1]
    xy = B[0, 1]
    yx = B[1, 0]
    return np.array(
        [
            np.real(xx + yy),
            np.real(xx - yy),
            np.real(xy + yx),
            np.real(-1j * (xy - yx)),
        ],
        dtype=float,
    )



def matrix_sqrt_psd(matrix):
    eigvals, eigvecs = np.linalg.eigh(matrix)
    eigvals = np.clip(eigvals, 0.0, None)
    return eigvecs @ np.diag(np.sqrt(eigvals)) @ np.conjugate(eigvecs.T)



def sample_field_from_brightness(B, n=20000, seed=0):
    rng = np.random.default_rng(seed)
    sqrt_B = matrix_sqrt_psd(B)
    z = (rng.normal(size=(2, n)) + 1j * rng.normal(size=(2, n))) / np.sqrt(2.0)
    return sqrt_B @ z



def estimate_visibility(vp, vq):
    return (vp @ np.conjugate(vq.T)) / vp.shape[1]



def geometric_phase(u, v, l, m):
    return np.exp(-2j * np.pi * (u * l + v * m))



def rime_point_source(u, v, B, l=0.0, m=0.0, Jp=None, Jq=None):
    if Jp is None:
        Jp = np.eye(2, dtype=complex)
    if Jq is None:
        Jq = np.eye(2, dtype=complex)
    Kpq = geometric_phase(u, v, l, m)
    return Jp @ (Kpq * B) @ np.conjugate(Jq.T)



def rime_discrete_sky(u, v, sky, Jp=None, Jq=None):
    Vpq = np.zeros((2, 2), dtype=complex)
    for source in sky:
        B = brightness_matrix(source['I'], source.get('Q', 0.0), source.get('U', 0.0), source.get('V', 0.0))
        Vpq += rime_point_source(u, v, B, l=source['l'], m=source['m'], Jp=Jp, Jq=Jq)
    return Vpq



def mueller_from_jones(J):
    M = np.zeros((4, 4), dtype=float)
    for col, stokes_basis in enumerate(np.eye(4)):
        Bout = J @ brightness_matrix(*stokes_basis) @ np.conjugate(J.T)
        M[:, col] = stokes_from_brightness(Bout)
    return M



def show_matrix(ax, matrix, title, mode='real', cmap='coolwarm'):
    if mode == 'real':
        data = np.real(matrix)
    elif mode == 'imag':
        data = np.imag(matrix)
    elif mode == 'abs':
        data = np.abs(matrix)
    else:
        raise ValueError('mode must be real, imag or abs')

    im = ax.imshow(data, cmap=cmap)
    ax.set_title(title)
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(['X', 'Y'])
    ax.set_yticklabels(['X', 'Y'])
    for iy in range(2):
        for ix in range(2):
            ax.text(ix, iy, f'{data[iy, ix]:+.3f}', ha='center', va='center', color='black')
    return im

### 7.2.1 二元干涉仪、相关矩阵与可见度矩阵

设一个二元干涉仪由两台天线 $p$ 和 $q$ 构成。对每台天线，我们都记录两个正交极化通道的复电压，因此有电压向量

$$
\mathbf{v}_p = \mathbf{J}_p \mathbf{e}, \qquad \mathbf{v}_q = \mathbf{J}_q \mathbf{e},
$$

其中 $\mathbf{e}$ 是来自某一方向的源电场，$\mathbf{J}_p,\mathbf{J}_q$ 是两台天线各自的 Jones 链。相关器对这两个电压向量做时间平均，得到可见度矩阵

$$
\mathbf{V}_{pq} = \langle \mathbf{v}_p \mathbf{v}_q^H \rangle.
$$

把上一节的亮度矩阵 $\mathbf{B}=\langle \mathbf{e}\mathbf{e}^H\rangle$ 代入后，就得到最基本的矩阵关系：

$$
\mathbf{V}_{pq} = \mathbf{J}_p \, \mathbf{B} \, \mathbf{J}_q^H.
$$

下面用随机电场样本直接模拟一次“相关器估计”，看看它与解析表达是否一致。

In [ ]:
sourceBrightness = brightness_matrix(1.0, 0.20, -0.10, 0.05)
Jp = np.array(
    [[1.05 * np.exp(0.10j), 0.02j], [-0.01j, 0.95 * np.exp(-0.05j)]],
    dtype=complex,
)
Jq = np.array(
    [[0.98 * np.exp(-0.08j), -0.015j], [0.03j, 1.02 * np.exp(0.12j)]],
    dtype=complex,
)

fieldSamples = sample_field_from_brightness(sourceBrightness, n=40000, seed=2)
voltageP = Jp @ fieldSamples
voltageQ = Jq @ fieldSamples
estimatedVisibility = estimate_visibility(voltageP, voltageQ)
modelVisibility = Jp @ sourceBrightness @ np.conjugate(Jq.T)

print('Source brightness matrix B =')
print(np.round(sourceBrightness, 4))
print()
print('Analytic visibility Vpq =')
print(np.round(modelVisibility, 4))
print()
print('Estimated visibility from voltage samples =')
print(np.round(estimatedVisibility, 4))
print()
print(f'max |V_est - V_model| = {np.max(np.abs(estimatedVisibility - modelVisibility)):.4e}')

fig, axes = plt.subplots(1, 3, figsize=(12.5, 4.0))
show_matrix(axes[0], sourceBrightness, 'Re(B)', mode='real')
show_matrix(axes[1], modelVisibility, 'Re(V model)', mode='real')
show_matrix(axes[2], estimatedVisibility - modelVisibility, '|V est - V model|', mode='abs', cmap='viridis')
plt.tight_layout()

这里最值得建立的直觉是：相关器输出并不是“再现原始 Jones 向量”，而是输出一个二阶统计对象，也就是可见度矩阵。它的每个元素都对应某一对极化通道之间的互相关。因此，RIME 从一开始就不是标量方程，而天然是矩阵方程。后面若只处理 Stokes $I$ 或标量增益，那只是特殊情形的进一步简化。

### 7.2.2 单点源的几何相位与基本 RIME

若天空中只有一个点源，且在小视场近似下只考虑方向余弦 $(l,m)$，则几何项可写成一个纯相位标量

$$
K_{pq}(l,m) = e^{-2\pi i(ul + vm)}.
$$

于是最基本的单点源 RIME 就可以写成

$$
\mathbf{V}_{pq} = \mathbf{J}_p \, K_{pq} \, \mathbf{B} \, \mathbf{J}_q^H.
$$

若暂时忽略所有非几何 Jones 项，则一个未偏振点源只会给可见度带来振幅恒定、相位随基线和源位置变化的条纹。

In [ ]:
uTrack = np.linspace(-40.0, 40.0, 500)
pointSources = {
    'Phase centre': 0.00,
    'Offset l=0.03': 0.03,
    'Offset l=-0.07': -0.07,
}

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for label, lcoord in pointSources.items():
    visibility = np.array([rime_point_source(u, 0.0, brightness_matrix(1.0), l=lcoord)[0, 0] for u in uTrack])
    axes[0].plot(uTrack, np.real(visibility), linewidth=2, label=label)
    axes[1].plot(uTrack, np.unwrap(np.angle(visibility)), linewidth=2, label=label)

axes[0].set_title('Real visibility of a point source')
axes[0].set_xlabel('u (wavelengths)')
axes[0].set_ylabel('Re(Vxx)')
axes[0].grid(alpha=0.25)
axes[0].legend(fontsize=9)

axes[1].set_title('Unwrapped phase')
axes[1].set_xlabel('u (wavelengths)')
axes[1].set_ylabel('phase [rad]')
axes[1].grid(alpha=0.25)
axes[1].legend(fontsize=9)

plt.tight_layout()

fringeSpacing = 1.0 / 0.03
print(f'l = 0.03 时，条纹周期约为 Δu = 1/l = {fringeSpacing:.2f} 个波长')

这个公式把几何和系统效应分开得非常清楚：

* $K_{pq}$ 负责告诉我们源偏离相位中心时，可见度会如何产生相位旋转；
* $\mathbf{J}_p, \mathbf{J}_q$ 负责告诉我们仪器和传播介质怎样进一步改变这个理想信号；
* $\mathbf{B}$ 负责给出源本身的总强度与偏振结构。

在文献中，人们有时把这种针对某一具体观测配置写出的方程称为 **a RIME**，而把能够容纳任意 Jones 链与任意天空项的统一框架称为 **the RIME**。这两者的差别，不在数学形式是否“更高级”，而在模型是否已经针对具体仪器展开。

### 7.2.3 离散天空求和与全天 RIME

若天空中有多个点源，则总可见度就是各个方向贡献的线性叠加：

$$
\mathbf{V}_{pq} = \sum_s \mathbf{J}_{p,s} \, K_{pq,s} \, \mathbf{B}_s \, \mathbf{J}_{q,s}^H.
$$

当点源越来越密，这个离散和就过渡到全天积分形式，也就是常见的全天 RIME。下面先看离散求和的版本，因为它最能直接展示“每个源都在可见度平面上留下自己的条纹，而总信号只是这些条纹的矢量和”。

In [ ]:
singleSky = [{'I': 1.0, 'l': 0.00, 'm': 0.0}]
multiSky = [
    {'I': 1.00, 'l': 0.00, 'm': 0.0},
    {'I': 0.55, 'l': 0.04, 'm': 0.0},
    {'I': 0.35, 'l': -0.08, 'm': 0.0},
]

singleVisibility = np.array([rime_discrete_sky(u, 0.0, singleSky)[0, 0] for u in uTrack])
multiVisibility = np.array([rime_discrete_sky(u, 0.0, multiSky)[0, 0] for u in uTrack])
componentVisibilities = [
    np.array([rime_discrete_sky(u, 0.0, [src])[0, 0] for u in uTrack])
    for src in multiSky
]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].plot(uTrack, np.real(singleVisibility), linewidth=2, label='Single source')
axes[0].plot(uTrack, np.real(multiVisibility), linewidth=2, label='Three-source sky')
axes[0].set_title('Real part of visibility')
axes[0].set_xlabel('u (wavelengths)')
axes[0].set_ylabel('Re(Vxx)')
axes[0].grid(alpha=0.25)
axes[0].legend(fontsize=9)

for idx, vis in enumerate(componentVisibilities, start=1):
    axes[1].plot(uTrack, np.abs(vis), linewidth=1.5, label=f'Component {idx}')
axes[1].plot(uTrack, np.abs(multiVisibility), color='k', linewidth=2.2, label='Vector sum')
axes[1].set_title('Amplitude of component visibilities')
axes[1].set_xlabel('u (wavelengths)')
axes[1].set_ylabel('|Vxx|')
axes[1].grid(alpha=0.25)
axes[1].legend(fontsize=9)

plt.tight_layout()

print('u=10 wavelengths 处三源天空的 Vxx =', np.round(rime_discrete_sky(10.0, 0.0, multiSky)[0, 0], 4))
print('多源总流量之和                   =', sum(src['I'] for src in multiSky))

真正的全天 RIME 只是在此基础上把离散求和替换成连续积分，并在每个方向上允许出现方向相关 Jones 项：

$$
\mathbf{V}_{pq} = \iint \mathbf{J}_p(l,m) \, \mathbf{B}(l,m) \, \mathbf{J}_q^H(l,m) \, e^{-2\pi i(ul+vm)} \, dl \, dm.
$$

这也是为什么主波束、传播介质和宽场效应会让成像与校准迅速变复杂：一旦 $\mathbf{J}$ 依赖于方向，天空就不再只是一个简单的标量傅里叶变换对象，而是一个被方向相关矩阵逐点调制过的矩阵场。

### 7.2.4 Jones 语言与 Mueller 语言

在某些场景下，我们更希望直接讨论 Stokes 向量如何被系统混合，而不是继续保留亮度矩阵形式。这时就可以把 Jones 作用改写成 Mueller 作用：

$$
\mathbf{s}_{\mathrm{out}} = \mathbf{M}\,\mathbf{s}_{\mathrm{in}},
$$

其中 $\mathbf{s}=[I,Q,U,V]^T$，$\mathbf{M}$ 是一个 $4\times4$ 的 Mueller 矩阵。Mueller 语言在讨论偏振混合、泄漏和仪器极化时特别方便，但它的本质并没有脱离 Jones 语言，只是把同一个线性作用改写到了 Stokes 空间。

In [ ]:
jonesExample = np.array(
    [[1.0, 0.08j], [-0.05j, 0.93 * np.exp(0.20j)]],
    dtype=complex,
)
muellerExample = mueller_from_jones(jonesExample)
inputStokes = np.array([1.0, 0.20, -0.30, 0.10])
outputBrightness = jonesExample @ brightness_matrix(*inputStokes) @ np.conjugate(jonesExample.T)
outputStokesFromJones = stokes_from_brightness(outputBrightness)
outputStokesFromMueller = muellerExample @ inputStokes

print('Input Stokes              =', np.round(inputStokes, 4))
print('Output from Jones action  =', np.round(outputStokesFromJones, 4))
print('Output from Mueller form  =', np.round(outputStokesFromMueller, 4))
print(f'max |difference|          = {np.max(np.abs(outputStokesFromJones - outputStokesFromMueller)):.3e}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
im = axes[0].imshow(muellerExample, cmap='coolwarm')
axes[0].set_title('Mueller matrix derived from J')
axes[0].set_xticks(range(4))
axes[0].set_yticks(range(4))
axes[0].set_xticklabels(['I', 'Q', 'U', 'V'])
axes[0].set_yticklabels(['I', 'Q', 'U', 'V'])
for iy in range(4):
    for ix in range(4):
        axes[0].text(ix, iy, f'{muellerExample[iy, ix]:+.2f}', ha='center', va='center', color='black', fontsize=9)
fig.colorbar(im, ax=axes[0], shrink=0.85)

x = np.arange(4)
width = 0.35
axes[1].bar(x - width / 2, inputStokes, width=width, label='Input')
axes[1].bar(x + width / 2, outputStokesFromMueller, width=width, label='Output')
axes[1].set_xticks(x)
axes[1].set_xticklabels(['I', 'Q', 'U', 'V'])
axes[1].set_ylabel('Stokes value')
axes[1].set_title('Stokes mixing in Mueller form')
axes[1].grid(axis='y', alpha=0.25)
axes[1].legend()

plt.tight_layout()

到这里可以把第 7.1 与第 7.2 连成一条完整主线：Jones 向量描述相干电场，亮度矩阵描述源的统计极化，RIME 把 Jones 链和亮度矩阵接到可见度矩阵，而 Mueller 形式则把同一作用改写成 Stokes 空间中的混合。后面的 7.3 到 7.8 节，就是把这些数学对象逐一对应到真实观测系统中的放大器、相关器、主波束、馈源、传播介质和 RFI 上。

***

* 下一节： [7.3 模拟电子学](7_3_analogue.ipynb)